# Universal Multi-Market (3,000+ Assets) Normalization & Feature Pipeline

### Institutional Quantitative Architecture
This notebook is an end-to-end quantitative engineering pipeline designed to ingest, clean, harmonize, filter, and feature-engineer large-scale heterogeneous market data feeds (**3,000+ assets** across **US Equities**, **Saudi Tadawul**, **Forex**, and **Crypto**) from raw **CSV/Parquet** into institutional feature stores.

```
Raw Multi-Market CSV / Parquet (3,000+ Assets)
  │
  ├── [Stage 1] High-Throughput Chunked Ingestion & RAM Downcasting (float32, category)
  ├── [Stage 2] Schema Harmonization & UTC Timezone Normalization
  ├── [Stage 3] Calendar Alignment & Session Masking (Tadawul Sun-Thu vs US vs FX vs Crypto)
  ├── [Stage 4] Integrity Audits, De-duplication & Bad Tick / Flash Crash Filtering (Rolling MAD)
  ├── [Stage 5] Cross-Sectional Universe Filtering (ADV_$ Thresholds, Penny Stock Trimming)
  ├── [Stage 6] Advanced Financial Transformations:
  │     • Multi-Horizon Log Returns (1d, 5d, 20d)
  │     • Garman-Klass Volatility & Rolling Realized Volatility
  │     • Amihud Microstructure Illiquidity Ratio
  │     • Cross-Sectional Rank & Z-Score Normalization (Per Timestamp t)
  │     • Marcos López de Prado Memory-Preserving Fractional Differentiation (d=0.40)
  └── [Stage 7] Partitioned Parquet Lake Export (market/year_month) & Sub-Second DuckDB Analytics
```


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display, HTML

print("Environment initialized: pandas", pd.__version__, "| duckdb", duckdb.__version__)


## Stage 1 & 2: Ingestion, Memory Optimization & Schema Harmonization
Ingests 3,000 tickers across multiple venues. Downcasts `float64` to `float32`, casts symbols to `category`, and maps disparate headers into the standard schema: `[timestamp, symbol, market, currency, open, high, low, close, volume]`.

In [ ]:
# Synthesize or Ingest 3,000 Multi-Market Assets
np.random.seed(42)
n_assets = 3000
n_days = 60
dates = pd.date_range(end=pd.Timestamp.now(tz='UTC').floor('D'), periods=n_days, freq='B')

print(f"[*] Ingesting raw multi-market feed ({n_assets:,} symbols across US Equities, Tadawul, FX, Crypto)...")
from notebooks.universal_multi_asset_normalization_pipeline import (
    generate_synthetic_multi_market_feed,
    optimize_memory,
    normalize_schema,
    clean_and_audit_data,
    apply_universe_filters,
    engineer_features,
    export_to_parquet_lake
)

raw_df = generate_synthetic_multi_market_feed(n_assets=n_assets, n_days=n_days)
mem_df = optimize_memory(raw_df)
schema_df = normalize_schema(mem_df)
display(schema_df.head(5))


## Stage 3 & 4: Data Quality Audit, De-duplication & Bad Tick Flash Crash Filters
Catches exchange glitches, duplicate rows, negative volume anomalies, and flash crashes via Median Absolute Deviation (MAD) filtering.

In [ ]:
clean_df = clean_and_audit_data(schema_df)
display(clean_df.describe().round(2))


## Stage 5: Cross-Sectional Universe Filtering
Filters the 3,000 assets by institutional viability: **Average Daily Dollar Volume ($ADV_\$ \ge \$250,000$)**, **Minimum Price ($\ge \$3.00$)**, and **Minimum Coverage ($\ge 85\%$)** to eliminate illiquid penny stocks.

In [ ]:
univ_df = apply_universe_filters(clean_df, min_adv_dollar=250000.0, min_price=3.0)
print(f"Retained {univ_df['symbol'].nunique():,} institutional assets across {len(univ_df):,} bars.")


## Stage 6: Institutional Feature Engineering & Cross-Sectional Normalization
Computes:
- Multi-horizon log returns ($r_{1d}, r_{5d}$)
- Garman-Klass Volatility: $\sigma_{GK}^2 = 0.5 (\ln(H/L))^2 - (2\ln 2 - 1)(\ln(C/O))^2$
- Amihud Illiquidity Ratio: $\text{ILLIQ} = \frac{|r_t|}{\text{DollarVol}_t} \times 10^6$
- Cross-Sectional Z-Score & Rank (per timestamp $t$ across universe)
- Marcos López de Prado Fractional Differentiation ($d=0.40$)

In [ ]:
features_df = engineer_features(univ_df)
display(features_df[["symbol", "timestamp", "close", "log_ret_1d", "garman_klass_vol", "amihud_illiq", "cs_zscore_1d", "cs_rank_1d", "frac_diff_px"]].head(10))


## Stage 7: Partitioned Parquet Lake Export & DuckDB Sub-Second Queries
Writes the final dataset partitioned by `market` and `year_month` using ZSTD compression. Executes lightning-fast analytics using DuckDB.

In [ ]:
summary = export_to_parquet_lake(features_df)
display(summary)
print("Universal 3,000+ Asset Normalization Pipeline Complete!")
